<a href="https://colab.research.google.com/github/saumyabaranwal/visionmate/blob/main/currency_detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install roboflow torch torchvision onnx onnxscript onnxruntime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 120.6 MB/s eta 0:00:00


In [30]:
!unzip -q my_currency.zip -d my_currency_extracted
!ls my_currency_extracted/my_currency

1  10  100  20	200  5	50  500


In [31]:
import os, shutil

my_photos_dir = "my_currency_extracted/my_currency"
valid_classes = ["10", "20", "50", "100", "200", "500"]

for class_name in valid_classes:
    src_dir = f"{my_photos_dir}/{class_name}"
    if not os.path.exists(src_dir):
        print(f"Skipping {class_name}")
        continue
    target_dir = f"currency_final/{class_name}"
    for fname in os.listdir(src_dir):
        shutil.copy(f"{src_dir}/{fname}", f"{target_dir}/myphoto_{fname}")

for class_name in sorted(os.listdir("currency_final"), key=lambda x: int(x)):
    count = len(os.listdir(f"currency_final/{class_name}"))
    print(f"{class_name}: {count}")

10: 4698
20: 4408
50: 3753
100: 4309
200: 45
500: 4058
2000: 35


In [44]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import os

# 1. Confirm your photos still exist
my_photos_dir = "my_currency_extracted/my_currency"
print("Photos folder exists:", os.path.exists(my_photos_dir))

# 2. Build a clean-only-your-photos dataset (no train/val split, too little data for that)
transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(my_photos_dir, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
class_names = dataset.classes
print("Classes:", class_names)
print("Total images:", len(dataset))

# 3. Fresh model, fine-tune ONLY on your photos
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = models.mobilenet_v2(weights="IMAGENET1K_V1")
for param in model.features.parameters():
    param.requires_grad = False

num_classes = len(class_names)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

# 4. Train for more epochs since dataset is tiny (needs more passes to learn anything)
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss:.3f} | Train Acc: {acc:.2f}%")

print("Done training on your photos only")

Photos folder exists: True
Classes: ['1', '10', '100', '20', '200', '5', '50', '500']
Total images: 101
Device: cuda
Epoch 1/30 | Loss: 27.830 | Train Acc: 21.78%
Epoch 5/30 | Loss: 11.764 | Train Acc: 86.14%
Epoch 10/30 | Loss: 7.287 | Train Acc: 89.11%
Epoch 15/30 | Loss: 5.184 | Train Acc: 92.08%
Epoch 20/30 | Loss: 4.042 | Train Acc: 94.06%
Epoch 25/30 | Loss: 2.964 | Train Acc: 95.05%
Epoch 30/30 | Loss: 2.741 | Train Acc: 97.03%
Done training on your photos only


In [45]:
import json
with open("class_names_mine.json", "w") as f:
    json.dump(class_names, f)

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(model, dummy_input, "currency_mine.onnx",
    input_names=["input"], output_names=["output"], opset_version=12, dynamo=False)

from google.colab import files
files.download("currency_mine.onnx")
files.download("class_names_mine.json")

You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>